# The website, on Colab's GPU

Runs `app/app.py` on a Colab GPU and gives you a public link. Open the link in a new tab, type a prompt, drag the switch slider, press **Generate**. Keep this tab open while you use it; closing it stops the server.

Setup: Runtime > Change runtime type > **T4 GPU** > Save. Then Runtime > Run all. The first Generate click loads the models (~10 GB download, 3-5 minutes); later clicks take about 20-40 seconds per image on a T4.

In [ ]:
# --- Setup (Colab: Runtime > Change runtime type > T4 GPU or better) ---
import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
REPO_URL = ""   # paste your GitHub URL here once the repo is pushed; leave empty to upload the zip instead
if IN_COLAB and not os.path.exists("image-rerouter"):
    if REPO_URL:
        !git clone -q $REPO_URL
    else:
        from google.colab import files
        import glob, shutil
        print("Pick the image-rerouter zip (any name, e.g. image-rerouter_2026-10-26.zip) in the file dialog below.")
        ZIP = next(iter(files.upload()))
        !rm -rf /content/_unzip && unzip -q -o "$ZIP" -d /content/_unzip && rm "$ZIP"
        # the code is wherever config.yaml sits next to rerouter/, whatever the zip's folder is called
        src = next(os.path.dirname(c) for c in glob.glob("/content/_unzip/**/config.yaml", recursive=True)
                   if os.path.isdir(os.path.join(os.path.dirname(c), "rerouter")))
        shutil.move(src, "/content/image-rerouter")
if IN_COLAB:
    %cd image-rerouter
    !pip install -q -r requirements.txt
    !pip uninstall -q -y torchao   # Colab's preinstalled torchao is too old for new diffusers (FqnToConfig import error); unused here
# optional: faster downloads + higher rate limits
# from huggingface_hub import login; login()
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
from rerouter import utils
cfg = utils.load_config()
cfg

In [ ]:
APP_PASSWORD = ""   # set one so only people you give it to can use your GPU (user name: guest); blank = no login
os.environ["RR_AUTH"] = f"guest:{APP_PASSWORD}" if APP_PASSWORD else ""
!python app/app.py --share